# Silver Table 15: project flood mapping

## Purpose

Classify every canonical DPWH project against the governed MGB flood-susceptibility polygons. For each project, the notebook checks whether its usable map point falls inside an approved, usable MGB polygon. It publishes the project's susceptibility level only when exactly one approved level is present at that point.

The table answers one question: **what flood susceptibility is present at this project's mapped point?**

It does **not** answer whether a project is a flood-control project. Flood-control identity comes from the project classification and the flood-list reconciliation in `silver_project_source_match`. The table also does not judge whether investment is sufficient, fair, effective, or adequate. It supplies flood-risk context only.

## Grain

One final project-to-flood classification result per canonical `silver_project` row per deterministic project-flood mapping run. Every canonical project produces exactly one row. This includes projects without a usable coordinate, projects with no MGB match, and projects that are ambiguous across approved levels.

Required uniqueness: `(source_system, contract_id, run_id)`. The primary key is `project_flood_map_key`.

## Source authority

| Input | Role |
| --- | --- |
| `02-silver.silver_project` | Canonical project identity, parsed coordinates, and project lineage |
| `01-bronze.flood_susceptibility` | Selected MGB flood-area rows, raw susceptibility codes, and `geometry_json` |
| `02-silver.config_mgb_susceptibility_mapping` | The only authority for code-to-level and severity-rank standardization |
| `01-bronze.load_log` and validation results | Compact upstream safety, timeliness, and publisher-version evidence |

Table 15 does not reread Bronze DPWH business rows or redo project consolidation. It does not read population, flood-control components, Table 13, Table 14, or Gold. It never infers susceptibility from a project category, description, or regional aggregate.

## 1. Confirm upstream safety and select versions

**What:** declare every rule constant once, prove the newest terminal MGB and DPWH loads are safe, and require safe Bronze, project, and CONFIG validation evidence. Then select one project run, one MGB snapshot, and one approved mapping version, and derive one deterministic run ID.

**Why:** the newest audit event wins, so an older success cannot hide a newer failed or unfinished load. `silver_project` must be built from the newest validated DPWH snapshot, so the table never classifies stale projects. Only compact audit metadata and lineage columns are read here.

**Run identity:** the run ID hashes these contracts:

- the project run, snapshot, and ingest run
- the MGB snapshot, ingest run, and optional publisher version
- the mapping version and its source contract
- the classification rule and rule version

No date, timestamp, UUID, or random value enters logical identity. `classification_rule_version` also carries the predicate and the coordinate screen, so a changed screen cannot silently reuse an old run ID.

**Coordinate contract:** the latitude and longitude screen is the Table 13 contract, copied once from `src/config.py`. It is declared once here and reused everywhere below.

**Protects:** one project run, one MGB snapshot, one approved mapping version, and complete lineage.

**Expected validation:** the validator repeats these timeliness checks against the published lineage and reproduces the run ID.

In [ ]:
%sql
USE CATALOG `buildabida-capstone`;
CREATE SCHEMA IF NOT EXISTS `02-silver`;

-- every rule constant is declared once here and referenced by name below
DECLARE OR REPLACE VARIABLE project_source_system STRING DEFAULT 'DPWH';
DECLARE OR REPLACE VARIABLE mgb_mapping_source_system STRING DEFAULT 'DENR MGB flood susceptibility';
DECLARE OR REPLACE VARIABLE source_geometry_srid INT DEFAULT 4326;
-- table 13 coordinate screen, copied once from ph_lat and ph_lon in src/config.py
DECLARE OR REPLACE VARIABLE min_latitude DOUBLE DEFAULT 4.2;
DECLARE OR REPLACE VARIABLE max_latitude DOUBLE DEFAULT 21.3;
DECLARE OR REPLACE VARIABLE min_longitude DOUBLE DEFAULT 116.0;
DECLARE OR REPLACE VARIABLE max_longitude DOUBLE DEFAULT 127.0;
-- performance-only candidate grid in degrees, it never changes which polygons match
DECLARE OR REPLACE VARIABLE candidate_grid_cell_degrees DOUBLE DEFAULT 0.1;
DECLARE OR REPLACE VARIABLE classification_rule STRING
    DEFAULT 'POINT_INTERSECTS_APPROVED_USABLE_MGB_POLYGON|UNIQUE_APPROVED_LEVEL_REQUIRED|NO_SEVERITY_PRECEDENCE';
DECLARE OR REPLACE VARIABLE classification_rule_version STRING;

SET VAR classification_rule_version = CONCAT(
    'silver-project-flood-map-v2|predicate=ST_INTERSECTS',
    '|coordinate-screen-latitude=', min_latitude, ':', max_latitude,
    '|coordinate-screen-longitude=', min_longitude, ':', max_longitude
);

-- newest terminal audit event per source, including failed or unfinished loads
CREATE OR REPLACE TEMPORARY VIEW latest_project_flood_source_load AS
SELECT * EXCEPT (terminal_rank)
FROM (
    SELECT *, ROW_NUMBER() OVER (
        PARTITION BY table_name
        ORDER BY COALESCE(completed_at, started_at) DESC, started_at DESC
    ) AS terminal_rank
    FROM `01-bronze`.load_log
    WHERE table_name IN ('flood_susceptibility', 'dpwh_projects')
      AND status IN ('STARTED', 'SUCCESS', 'FAILED', 'SKIPPED_IDEMPOTENT')
)
WHERE terminal_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW latest_project_flood_bronze_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT dq.*, DENSE_RANK() OVER (
        PARTITION BY dq.table_name ORDER BY dq.run_ts DESC, dq.run_id DESC
    ) AS validation_rank
    FROM `04-validation`.dq_results AS dq
    INNER JOIN latest_project_flood_source_load AS latest
        ON dq.table_name = latest.table_name
       AND dq.snapshot_id = latest.snapshot_id
)
WHERE validation_rank = 1;

CREATE OR REPLACE TEMPORARY VIEW selected_project_validation AS
SELECT *
FROM `04-validation`.silver_dq_results
WHERE table_name IN ('silver_project', 'silver_dpwh_project_component')
  AND pipeline_run_id = (SELECT MAX(run_id) FROM `02-silver`.silver_project);

CREATE OR REPLACE TEMPORARY VIEW latest_config_validation AS
SELECT * EXCEPT (validation_rank)
FROM (
    SELECT *, DENSE_RANK() OVER (ORDER BY run_ts DESC, run_id DESC) AS validation_rank
    FROM `04-validation`.silver_config_dq_results
    WHERE layer = 'silver_config'
)
WHERE validation_rank = 1;

SELECT ASSERT_TRUE(
    COUNT(*) = 2
    AND COUNT_IF(status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')) = 2
    AND COUNT_IF(snapshot_id IS NOT NULL AND TRIM(snapshot_id) <> '') = 2
    AND COUNT_IF(rows_loaded > 0) = 2,
    'STOP: latest MGB or DPWH Bronze load is missing, failed, unidentified, or empty'
)
FROM latest_project_flood_source_load;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT table_name) = 2
    AND COUNT_IF(action = 'stop' AND status IN ('FAIL', 'ERROR')) = 0,
    'STOP: latest MGB or DPWH Bronze validation is missing or unsafe'
)
FROM latest_project_flood_bronze_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status = 'FAIL') = 0,
    'STOP: selected Silver project validation is missing or unsafe'
)
FROM selected_project_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0 AND COUNT_IF(action = 'stop' AND status = 'FAIL') = 0,
    'STOP: latest Silver configuration validation is missing or unsafe'
)
FROM latest_config_validation;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT run_id) = 1
    AND COUNT(DISTINCT source_snapshot_id) = 1
    AND COUNT(DISTINCT source_ingest_run_id) = 1
    AND COUNT(*) = COUNT(DISTINCT project_key)
    AND MAX(source_snapshot_id) = (
        SELECT snapshot_id FROM latest_project_flood_source_load
        WHERE table_name = 'dpwh_projects'
    ),
    'STOP: silver_project must be nonempty, unique, and built from the latest validated DPWH snapshot'
)
FROM `02-silver`.silver_project;

-- lineage columns only, so this does not parse the large geometry field
SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT(DISTINCT _source_snapshot_id) = 1
    AND COUNT(DISTINCT _ingest_run_id) = 1
    AND MAX(_source_snapshot_id) = (
        SELECT snapshot_id FROM latest_project_flood_source_load
        WHERE table_name = 'flood_susceptibility'
    ),
    'STOP: current MGB Bronze rows do not match the validated MGB snapshot'
)
FROM `01-bronze`.flood_susceptibility;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT mapping.mapping_version) = 1
    AND COUNT(DISTINCT mapping.source_version) = 1,
    'STOP: exactly one active approved MGB mapping version and source contract is required'
)
FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
WHERE mapping.source_system = mgb_mapping_source_system
  AND mapping.approval_status = 'APPROVED'
  AND mapping.is_active = TRUE;

CREATE OR REPLACE TEMPORARY VIEW selected_project_flood_versions AS
WITH project_context AS (
    SELECT
        MAX(run_id) AS project_run_id,
        MAX(source_snapshot_id) AS project_source_snapshot_id,
        MAX(source_ingest_run_id) AS project_source_ingest_run_id,
        MAX(transformation_rule_version) AS project_rule_version
    FROM `02-silver`.silver_project
),
project_audit AS (
    -- load time and optional publisher label recorded by the load that created the project snapshot
    SELECT
        MAX(NULLIF(TRIM(log.source_version), '')) AS project_source_version,
        MAX(COALESCE(log.completed_at, log.started_at)) AS project_source_load_ts
    FROM `01-bronze`.load_log AS log
    INNER JOIN project_context AS context
        ON log.run_id = context.project_source_ingest_run_id
    WHERE log.table_name = 'dpwh_projects'
      AND log.status IN ('SUCCESS', 'SKIPPED_IDEMPOTENT')
),
mgb_context AS (
    SELECT
        MAX(_source_snapshot_id) AS mgb_source_snapshot_id,
        MAX(_ingest_run_id) AS mgb_source_ingest_run_id,
        MAX(_ingested_at) AS mgb_source_load_ts
    FROM `01-bronze`.flood_susceptibility
),
mgb_publisher AS (
    -- optional publisher label recorded by the load that created the selected rows
    SELECT MAX(NULLIF(TRIM(log.source_version), '')) AS mgb_source_version
    FROM `01-bronze`.load_log AS log
    INNER JOIN mgb_context AS context
        ON log.run_id = context.mgb_source_ingest_run_id
    WHERE log.table_name = 'flood_susceptibility'
      AND log.status = 'SUCCESS'
),
mapping_context AS (
    SELECT
        MAX(mapping.mapping_version) AS susceptibility_mapping_version,
        MAX(mapping.source_version) AS mgb_mapping_source_version
    FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
    WHERE mapping.source_system = mgb_mapping_source_system
      AND mapping.approval_status = 'APPROVED'
      AND mapping.is_active = TRUE
)
SELECT
    project_context.*, project_audit.*, mgb_context.*, mgb_publisher.*, mapping_context.*,
    classification_rule AS classification_rule,
    classification_rule_version AS classification_rule_version,
    SHA2(CONCAT_WS(
        '||',
        project_context.project_run_id, project_context.project_source_snapshot_id,
        project_context.project_source_ingest_run_id,
        mgb_context.mgb_source_snapshot_id, mgb_context.mgb_source_ingest_run_id,
        COALESCE(mgb_publisher.mgb_source_version, 'NO_PUBLISHER_VERSION'),
        mapping_context.susceptibility_mapping_version,
        mapping_context.mgb_mapping_source_version,
        classification_rule, classification_rule_version
    ), 256) AS project_flood_run_id
FROM project_context
CROSS JOIN project_audit
CROSS JOIN mgb_context
CROSS JOIN mgb_publisher
CROSS JOIN mapping_context;

SELECT ASSERT_TRUE(
    COUNT(*) = 1
    AND COUNT_IF(
        project_run_id IS NULL OR project_source_snapshot_id IS NULL
        OR project_source_ingest_run_id IS NULL OR project_rule_version IS NULL
        OR project_source_load_ts IS NULL
        OR mgb_source_snapshot_id IS NULL OR mgb_source_ingest_run_id IS NULL
        OR mgb_source_load_ts IS NULL
        OR susceptibility_mapping_version IS NULL OR mgb_mapping_source_version IS NULL
    ) = 0,
    'STOP: selected project-flood lineage is incomplete'
)
FROM selected_project_flood_versions;

SELECT * FROM selected_project_flood_versions;

## 2. Prepare canonical project points

**What:** read every canonical project once, assign the Table 13 coordinate status, and build a point only for a safe coordinate pair with a stable project identity.

**Why:** `silver_project` already publishes parsed coordinates, so Bronze coordinates are never cleaned again. `silver_project` has no coordinate-status column, so this step applies the exact Table 13 rule with the screen declared in Step 1:

| Status | Meaning |
| --- | --- |
| `MISSING_PAIR` | Latitude and longitude are both missing |
| `PARTIAL_PAIR` | Only one of the two values is present |
| `OUT_OF_RANGE` | The pair falls outside the Philippine screen |
| `VALID_PAIR` | The pair is present and inside the screen |

Points use longitude first, `ST_POINT(longitude, latitude, 4326)`, like Table 13. Each point also receives a candidate grid cell. The cell is a pure performance key and is explained in Step 4.

**Protects:** no point is created from an unsafe pair. No project is dropped. A project without a usable point keeps its row and later receives `UNMATCHED` with a controlled reason.

**Expected validation:** the validator recomputes the coordinate status independently and proves that only `VALID_PAIR` coordinates take part in spatial matching.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_flood_input AS
SELECT
    project_source_system AS source_system,
    project.project_key,
    project.contract_id,
    project.latitude_parsed AS latitude,
    project.longitude_parsed AS longitude,
    CASE
        WHEN project.latitude_parsed IS NULL AND project.longitude_parsed IS NULL THEN 'MISSING_PAIR'
        WHEN project.latitude_parsed IS NULL OR project.longitude_parsed IS NULL THEN 'PARTIAL_PAIR'
        WHEN project.latitude_parsed NOT BETWEEN min_latitude AND max_latitude
          OR project.longitude_parsed NOT BETWEEN min_longitude AND max_longitude THEN 'OUT_OF_RANGE'
        ELSE 'VALID_PAIR'
    END AS coordinate_status,
    project.project_key IS NOT NULL
        AND project.contract_id IS NOT NULL
        AND TRIM(project.contract_id) <> '' AS has_valid_identity
FROM `02-silver`.silver_project AS project;

-- only safe coordinates with a stable identity become points
CREATE OR REPLACE TEMPORARY VIEW project_flood_points AS
SELECT
    project_key,
    ST_POINT(longitude, latitude, source_geometry_srid) AS project_point,
    CAST(FLOOR(longitude / candidate_grid_cell_degrees) AS BIGINT) AS grid_x,
    CAST(FLOOR(latitude / candidate_grid_cell_degrees) AS BIGINT) AS grid_y
FROM project_flood_input
WHERE coordinate_status = 'VALID_PAIR'
  AND has_valid_identity;

SELECT
    coordinate_status,
    has_valid_identity,
    COUNT(*) AS project_rows
FROM project_flood_input
GROUP BY ALL
ORDER BY coordinate_status, has_valid_identity;

## 3. Reuse the governed MGB classification contract

**What:** read only active `APPROVED` mapping rows, then classify every selected MGB row once by mapping status and geometry status. Finally, reconcile every source row before any spatial work.

**Why:** Table 14 is the authority for MGB behavior. This step reuses its contract exactly:

- **Mapping:** configuration is the only standardization authority. There is no `CASE` mapping of LF, MF, HF, or VHF in this notebook. Each raw code maps to one level and one rank, and levels and ranks are one-to-one.
- **Parser:** the SQL below is the Table 14 parser, copied without change. It parses the geometry as published, then applies the targeted quoted-number normalization, then converts supported Esri `rings` JSON, reading counterclockwise rings as holes when the first reading is invalid. The first successful governed parse wins. A regression test keeps the two copies identical.
- **Geometry status:** `BLANK_GEOMETRY`, `UNPARSEABLE_GEOMETRY`, `EMPTY_GEOMETRY`, `UNSUPPORTED_GEOMETRY_TYPE`, `INVALID_GEOMETRY`, or `USABLE_GEOMETRY`. Only a nonempty, OGC-valid Polygon or MultiPolygon is usable. Invalid geometry is excluded and flagged, never repaired with `ST_BUFFER` or another hidden rule.

**Unresolved by design:** blank codes, `No rating`, and any other unapproved code remain `UNMAPPED`. They never become Low, Moderate, High, Very High, or `Unknown`. Gold may route unresolved relationships through its governed unknown member later. Silver never invents one.

**Source accounting (STOP):**

```text
selected MGB rows
= mapped approved + usable geometry
+ mapped approved + unusable geometry
+ unmapped or unrated + usable geometry
+ unmapped or unrated + unusable geometry
= audited Bronze rows_loaded
```

The run also stops if usable geometry is not in SRID 4326, or if mapped rows have geometry text but none parse.

**Source grain:** the MGB extract has no stable polygon business key, so none is invented. Lineage is the snapshot, ingest run, and versions, plus candidate counts per project.

**Expected validation:** the validator reparses the snapshot and reproduces the accounting.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW approved_mgb_susceptibility_mapping AS
SELECT
    UPPER(TRIM(mapping.raw_susceptibility_code)) AS raw_susceptibility_code,
    mapping.standardized_level,
    mapping.severity_rank,
    mapping.mapping_rule,
    mapping.mapping_version,
    mapping.source_version
FROM `02-silver`.config_mgb_susceptibility_mapping AS mapping
CROSS JOIN selected_project_flood_versions AS selected
WHERE mapping.source_system = mgb_mapping_source_system
  AND mapping.approval_status = 'APPROVED'
  AND mapping.is_active = TRUE
  AND mapping.mapping_version = selected.susceptibility_mapping_version;

SELECT ASSERT_TRUE(
    COUNT(*) > 0
    AND COUNT_IF(
        raw_susceptibility_code IS NULL OR raw_susceptibility_code = ''
        OR standardized_level IS NULL OR TRIM(standardized_level) = ''
        OR severity_rank IS NULL OR severity_rank <= 0
    ) = 0
    AND COUNT(*) = COUNT(DISTINCT raw_susceptibility_code),
    'STOP: approved MGB mapping is empty, incomplete, or maps one raw code more than once'
)
FROM approved_mgb_susceptibility_mapping;

SELECT ASSERT_TRUE(
    COUNT(DISTINCT standardized_level) = COUNT(DISTINCT severity_rank)
    AND COUNT(DISTINCT standardized_level)
        = COUNT(DISTINCT NAMED_STRUCT('level', standardized_level, 'rank', severity_rank)),
    'STOP: approved MGB levels and severity ranks are not one-to-one'
)
FROM approved_mgb_susceptibility_mapping;

-- the parser below is the table 14 parser, kept identical by a regression test
CREATE OR REPLACE TEMPORARY VIEW mgb_source_classified AS
WITH source_rows AS (
    SELECT
        NULLIF(UPPER(TRIM(mgb.flood_susceptibility_code)), '') AS raw_susceptibility_code,
        NULLIF(TRIM(mgb.geometry_json), '') AS geometry_text
    FROM `01-bronze`.flood_susceptibility AS mgb
    CROSS JOIN selected_project_flood_versions AS selected
    WHERE mgb._source_snapshot_id = selected.mgb_source_snapshot_id
),
direct_parse AS (
    SELECT
        esri_text.*,
        -- an Esri JSON row is never GeoJSON, so the GeoJSON readers skip it
        CASE
            WHEN esri_text.esri_rings_text IS NULL
            THEN TRY_TO_GEOMETRY(esri_text.geometry_text)
        END AS published_geometry
    FROM (
        SELECT
            source_rows.*,
            -- Esri JSON keeps its coordinates as text under the "rings" key
            NULLIF(REGEXP_EXTRACT(source_rows.geometry_text, '(?s)"rings"\\s*:\\s*(\\[.*\\])', 1), '') AS esri_rings_text
        FROM source_rows
    ) AS esri_text
),
esri_ring_source AS (
    SELECT
        direct_parse.*,
        -- first reading, built as text: every ring is its own polygon.
        -- Text avoids FROM_JSON arrays, which run out of memory on the largest MGB polygons.
        CASE
            WHEN esri_rings_text IS NULL THEN NULL
            WHEN esri_rings_text RLIKE '\\]\\s*\\]\\s*,\\s*\\[\\s*\\['
            THEN TRY_TO_GEOMETRY(CONCAT(
                '{"type":"MultiPolygon","coordinates":[',
                REGEXP_REPLACE(esri_rings_text, '\\]\\s*\\]\\s*,\\s*\\[\\s*\\[', ']]],[[['),
                ']}'
            ))
            ELSE TRY_TO_GEOMETRY(CONCAT('{"type":"Polygon","coordinates":', esri_rings_text, '}'))
        END AS esri_ring_geometry
    FROM direct_parse
),
esri_ring_checked AS (
    SELECT
        esri_ring_source.*,
        esri_ring_geometry IS NOT NULL AND ST_ISVALID(esri_ring_geometry) AS esri_ring_geometry_is_valid
    FROM esri_ring_source
),
esri_ring_arrays AS (
    SELECT
        esri_ring_checked.*,
        -- ring arrays are read only when the first reading is invalid, to find the holes
        CASE
            WHEN esri_rings_text IS NOT NULL AND NOT esri_ring_geometry_is_valid
            THEN FROM_JSON(esri_rings_text, 'ARRAY<ARRAY<ARRAY<DOUBLE>>>')
        END AS esri_rings
    FROM esri_ring_checked
),
esri_ring_parts AS (
    SELECT
        esri_ring_arrays.*,
        -- shoelace signed area per ring: Esri outer rings run clockwise (negative), holes counterclockwise
        TRANSFORM(
            esri_rings,
            ring -> AGGREGATE(
                SEQUENCE(1, SIZE(ring) - 1),
                CAST(0 AS DOUBLE),
                (area, i) -> area + ring[i - 1][0] * ring[i][1] - ring[i][0] * ring[i - 1][1]
            ) / 2
        ) AS esri_ring_signed_areas
    FROM esri_ring_arrays
),
esri_json_parsed AS (
    SELECT
        esri_ring_parts.*,
        CASE
            WHEN esri_ring_parts.esri_rings_text IS NOT NULL
            THEN ST_TRANSFORM(
                ST_SETSRID(
                    CASE
                        -- a valid first reading is kept as it is
                        WHEN esri_ring_geometry_is_valid
                        THEN esri_ring_geometry
                        -- otherwise read the counterclockwise rings as holes in the clockwise rings,
                        -- as the Esri polygon format defines them; this follows the format, it is not a repair
                        WHEN SIZE(FILTER(esri_ring_signed_areas, area -> area < 0)) > 0
                            AND SIZE(FILTER(esri_ring_signed_areas, area -> area > 0)) > 0
                        THEN ST_DIFFERENCE(
                            TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                                'type', 'MultiPolygon',
                                'coordinates', TRANSFORM(
                                    FILTER(
                                        ZIP_WITH(esri_rings, esri_ring_signed_areas, (ring, area) -> NAMED_STRUCT('ring', ring, 'area', area)),
                                        part -> part.area < 0
                                    ),
                                    part -> ARRAY(part.ring)
                                )
                            ))),
                            TRY_TO_GEOMETRY(TO_JSON(NAMED_STRUCT(
                                'type', 'MultiPolygon',
                                'coordinates', TRANSFORM(
                                    FILTER(
                                        ZIP_WITH(esri_rings, esri_ring_signed_areas, (ring, area) -> NAMED_STRUCT('ring', ring, 'area', area)),
                                        part -> part.area > 0
                                    ),
                                    part -> ARRAY(part.ring)
                                )
                            )))
                        )
                        ELSE esri_ring_geometry
                    END,
                    3857
                ),
                4326
            )
            ELSE NULL
        END AS esri_geometry
    FROM esri_ring_parts
),
normalized AS (
    SELECT
        esri_json_parsed.*,
        CASE
            WHEN esri_rings_text IS NULL
            THEN TRY_TO_GEOMETRY(REGEXP_REPLACE(
                REGEXP_REPLACE(geometry_text, '\"(\\[[^\"]*\\])\"', CONCAT(CHAR(36), '1')),
                '\"(-?[0-9]+(?:\\.[0-9]+)?(?:[Ee][+-]?[0-9]+)?)\"',
                CONCAT(CHAR(36), '1')
            ))
        END AS normalized_geometry
    FROM esri_json_parsed
),
parsed AS (
    SELECT
        normalized.*,
        COALESCE(
            published_geometry,
            normalized_geometry,
            esri_geometry
        ) AS mgb_geometry
    FROM normalized
)
SELECT
    parsed.raw_susceptibility_code,
    parsed.mgb_geometry,
    mapping.standardized_level AS flood_susceptibility_level,
    mapping.severity_rank,
    CASE
        WHEN mapping.raw_susceptibility_code IS NOT NULL THEN 'MAPPED_APPROVED'
        ELSE 'UNMAPPED'
    END AS susceptibility_mapping_status,
    CASE
        WHEN parsed.geometry_text IS NULL THEN 'NOT_PARSED'
        WHEN parsed.published_geometry IS NOT NULL THEN 'GEOJSON_AS_PUBLISHED'
        WHEN parsed.normalized_geometry IS NOT NULL THEN 'GEOJSON_QUOTED_NUMBERS_NORMALIZED'
        WHEN parsed.esri_geometry IS NOT NULL THEN 'ESRI_JSON_CONVERTED'
        ELSE 'PARSE_FAILED'
    END AS geometry_parse_method,
    CASE
        WHEN parsed.geometry_text IS NULL THEN 'BLANK_GEOMETRY'
        WHEN parsed.mgb_geometry IS NULL THEN 'UNPARSEABLE_GEOMETRY'
        WHEN ST_ISEMPTY(parsed.mgb_geometry) THEN 'EMPTY_GEOMETRY'
        WHEN REGEXP_REPLACE(UPPER(ST_GEOMETRYTYPE(parsed.mgb_geometry)), '^ST_', '')
            NOT IN ('POLYGON', 'MULTIPOLYGON') THEN 'UNSUPPORTED_GEOMETRY_TYPE'
        WHEN NOT ST_ISVALID(parsed.mgb_geometry) THEN 'INVALID_GEOMETRY'
        ELSE 'USABLE_GEOMETRY'
    END AS source_geometry_status
FROM parsed
LEFT JOIN approved_mgb_susceptibility_mapping AS mapping
    ON parsed.raw_susceptibility_code = mapping.raw_susceptibility_code;

-- one grouped scan both reports the accounting buckets and enforces the gate
SELECT
    accounting.*,
    ASSERT_TRUE(
        accounting.selected_mgb_rows = accounting.audited_rows_loaded
        AND accounting.selected_mgb_rows
            = accounting.mapped_usable_rows + accounting.mapped_unusable_rows
            + accounting.unmapped_usable_rows + accounting.unmapped_unusable_rows,
        'STOP: MGB source-row accounting does not reconcile'
    ) AS accounting_gate,
    ASSERT_TRUE(
        accounting.usable_wrong_srid_rows = 0,
        'STOP: usable MGB geometry is not in the expected source SRID'
    ) AS source_srid_gate,
    ASSERT_TRUE(
        accounting.mapped_geometry_text_rows = 0 OR accounting.mapped_usable_rows > 0,
        'STOP: no mapped MGB geometry is usable, so the geometry encoding contract is broken'
    ) AS encoding_gate
FROM (
    SELECT
        COUNT(*) AS selected_mgb_rows,
        MAX(latest.rows_loaded) AS audited_rows_loaded,
        COUNT_IF(susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status = 'USABLE_GEOMETRY') AS mapped_usable_rows,
        COUNT_IF(susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status <> 'USABLE_GEOMETRY') AS mapped_unusable_rows,
        COUNT_IF(susceptibility_mapping_status = 'UNMAPPED'
            AND source_geometry_status = 'USABLE_GEOMETRY') AS unmapped_usable_rows,
        COUNT_IF(susceptibility_mapping_status = 'UNMAPPED'
            AND source_geometry_status <> 'USABLE_GEOMETRY') AS unmapped_unusable_rows,
        COUNT_IF(susceptibility_mapping_status = 'MAPPED_APPROVED'
            AND source_geometry_status <> 'BLANK_GEOMETRY') AS mapped_geometry_text_rows,
        COUNT_IF(source_geometry_status = 'USABLE_GEOMETRY'
            AND ST_SRID(mgb_geometry) <> source_geometry_srid) AS usable_wrong_srid_rows,
        COUNT_IF(source_geometry_status = 'BLANK_GEOMETRY') AS blank_geometry_rows,
        COUNT_IF(source_geometry_status = 'UNPARSEABLE_GEOMETRY') AS unparseable_geometry_rows,
        COUNT_IF(source_geometry_status = 'EMPTY_GEOMETRY') AS empty_geometry_rows,
        COUNT_IF(source_geometry_status = 'UNSUPPORTED_GEOMETRY_TYPE') AS unsupported_geometry_rows,
        COUNT_IF(source_geometry_status = 'INVALID_GEOMETRY') AS invalid_geometry_rows,
        COUNT_IF(geometry_parse_method = 'GEOJSON_AS_PUBLISHED') AS parsed_as_published_rows,
        COUNT_IF(geometry_parse_method = 'GEOJSON_QUOTED_NUMBERS_NORMALIZED') AS parsed_after_normalization_rows,
        COUNT_IF(geometry_parse_method = 'ESRI_JSON_CONVERTED') AS esri_json_converted_rows
    FROM mgb_source_classified
    CROSS JOIN latest_project_flood_source_load AS latest
    WHERE latest.table_name = 'flood_susceptibility'
) AS accounting;

## 4. Build project-to-MGB spatial candidates

**What:** join usable project points to mapped, usable MGB polygons with `ST_INTERSECTS`, then aggregate the candidates to one row per project.

**Why:** this is the one expensive spatial operation in the table. Both sides are filtered first. Only `VALID_PAIR` points and only `MAPPED_APPROVED` plus `USABLE_GEOMETRY` polygons enter the join.

**Predicate:** `ST_INTERSECTS(polygon, point)` follows Table 14. A point on a polygon boundary counts as inside. There is no nearest polygon, no distance threshold, and no buffer.

**Candidate grid (performance only):** a spatial predicate alone gives the join no equality key. The engine could then compare every project with every polygon. To avoid that cross join, each polygon is listed under every grid cell its bounding box touches. Each point is listed under the one cell that contains it. The join then needs an equal cell before it tests `ST_INTERSECTS`.

The grid is exact. A polygon's bounding box contains every point the polygon intersects, and `FLOOR` keeps the point's cell inside the box's cell range. So the grid never removes a true match. A point sits in one cell, and a polygon appears at most once per cell, so no project-polygon pair is counted twice. The cell size is a tuning knob. It does not change results, so it is not part of the rule version or the run ID.

**Candidate evidence per project:**

- `matched_polygon_count`: source MGB rows whose polygon contains the point
- `matched_level_count`: distinct approved levels among those rows
- `candidate_levels`: those levels in sorted text order. The order is for display only and never implies precedence.

**Expected validation:** the validator rebuilds the candidates independently with a different grid size and proves the evidence matches.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW approved_mgb_candidate_cells AS
WITH approved_usable_polygons AS (
    SELECT
        mgb.flood_susceptibility_level,
        mgb.severity_rank,
        mgb.mgb_geometry,
        CAST(FLOOR(ST_XMIN(mgb.mgb_geometry) / candidate_grid_cell_degrees) AS BIGINT) AS grid_x_min,
        CAST(FLOOR(ST_XMAX(mgb.mgb_geometry) / candidate_grid_cell_degrees) AS BIGINT) AS grid_x_max,
        CAST(FLOOR(ST_YMIN(mgb.mgb_geometry) / candidate_grid_cell_degrees) AS BIGINT) AS grid_y_min,
        CAST(FLOOR(ST_YMAX(mgb.mgb_geometry) / candidate_grid_cell_degrees) AS BIGINT) AS grid_y_max
    FROM mgb_source_classified AS mgb
    WHERE mgb.susceptibility_mapping_status = 'MAPPED_APPROVED'
      AND mgb.source_geometry_status = 'USABLE_GEOMETRY'
)
-- one row per polygon and bounding-box grid cell
SELECT
    polygon.flood_susceptibility_level,
    polygon.severity_rank,
    polygon.mgb_geometry,
    INLINE(FLATTEN(TRANSFORM(
        SEQUENCE(polygon.grid_x_min, polygon.grid_x_max),
        x_cell -> TRANSFORM(
            SEQUENCE(polygon.grid_y_min, polygon.grid_y_max),
            y_cell -> NAMED_STRUCT('grid_x', x_cell, 'grid_y', y_cell)
        )
    )))
FROM approved_usable_polygons AS polygon;

-- candidates are aggregated to project grain before the final publish
CREATE OR REPLACE TEMPORARY VIEW project_flood_candidates AS
SELECT
    point.project_key,
    COUNT(*) AS matched_polygon_count,
    COUNT(DISTINCT polygon.flood_susceptibility_level) AS matched_level_count,
    SORT_ARRAY(COLLECT_SET(polygon.flood_susceptibility_level)) AS candidate_levels,
    CASE
        WHEN COUNT(DISTINCT polygon.flood_susceptibility_level) = 1
            THEN ELEMENT_AT(SORT_ARRAY(COLLECT_SET(polygon.flood_susceptibility_level)), 1)
    END AS unique_flood_susceptibility_level,
    CASE
        WHEN COUNT(DISTINCT polygon.flood_susceptibility_level) = 1
            THEN ELEMENT_AT(SORT_ARRAY(COLLECT_SET(polygon.severity_rank)), 1)
    END AS unique_severity_rank
FROM project_flood_points AS point
INNER JOIN approved_mgb_candidate_cells AS polygon
    ON point.grid_x = polygon.grid_x
   AND point.grid_y = polygon.grid_y
   AND ST_INTERSECTS(polygon.mgb_geometry, point.project_point)
GROUP BY point.project_key;

## 5. Resolve one final project flood classification

**What:** left join the project-grain candidates back to the complete project universe, then apply one conservative rule to every project.

**Why:** approved levels can overlap, as Table 14 measured. No approved business rule says Very High overrides High or that the highest severity wins. So this table never picks a level by severity, polygon order, or any other precedence.

| Case | Evidence | `match_status` | Published level and rank |
| --- | --- | --- | --- |
| A | Usable point, one or more polygons, exactly one distinct approved level | `MATCHED` | That level and its rank |
| B | Usable point, polygons from more than one distinct approved level | `AMBIGUOUS` | `NULL` |
| C | Usable point, no approved usable polygon | `UNMATCHED` | `NULL` |
| D | No usable point | `UNMATCHED` | `NULL` |
| None | Missing stable project identity | `INVALID_SOURCE` | `NULL` |

Case A stays `MATCHED` even when several overlapping polygons share the same level. The final level itself is not ambiguous, and `matched_polygon_count` keeps the real count.

**Controlled values:**

- `match_method`: `COORDINATE_MGB_POLYGON` when at least one polygon matched, otherwise `NONE`
- `match_quality`: `SPATIAL`, `REVIEW_REQUIRED`, or `UNRESOLVED`
- `ambiguity_reason`: `MULTIPLE_APPROVED_FLOOD_LEVELS`
- `exception_reason`: `MISSING_COORDINATE_PAIR`, `PARTIAL_COORDINATE_PAIR`, `COORDINATE_OUT_OF_RANGE`, `NO_APPROVED_MGB_INTERSECTION`, `MULTIPLE_APPROVED_FLOOD_LEVELS`, or `MISSING_STABLE_SOURCE_IDENTITY`

**Interpretation:** `NO_APPROVED_MGB_INTERSECTION` means only that the selected approved MGB polygons did not match the point. It is not Low risk, not zero risk, and not an `Unknown` susceptibility.

**Expected validation:** the validator proves each status agrees with its candidate counts, level, rank, and reason.

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW project_flood_decisions AS
WITH evidence AS (
    -- keep every canonical project even when no flood polygon matches
    SELECT
        project.*,
        COALESCE(candidate.matched_polygon_count, 0) AS matched_polygon_count,
        COALESCE(candidate.matched_level_count, 0) AS matched_level_count,
        COALESCE(candidate.candidate_levels, CAST(ARRAY() AS ARRAY<STRING>)) AS candidate_levels,
        candidate.unique_flood_susceptibility_level,
        candidate.unique_severity_rank
    FROM project_flood_input AS project
    LEFT JOIN project_flood_candidates AS candidate
        ON project.project_key = candidate.project_key
),
classified AS (
    SELECT
        evidence.*,
        CASE
            WHEN NOT evidence.has_valid_identity THEN 'INVALID_SOURCE'
            WHEN evidence.matched_level_count = 1 THEN 'MATCHED'
            WHEN evidence.matched_level_count > 1 THEN 'AMBIGUOUS'
            ELSE 'UNMATCHED'
        END AS match_status
    FROM evidence
)
SELECT
    classified.*,
    CASE
        WHEN classified.match_status IN ('MATCHED', 'AMBIGUOUS') THEN 'COORDINATE_MGB_POLYGON'
        ELSE 'NONE'
    END AS match_method,
    CASE classified.match_status
        WHEN 'MATCHED' THEN 'SPATIAL'
        WHEN 'AMBIGUOUS' THEN 'REVIEW_REQUIRED'
        ELSE 'UNRESOLVED'
    END AS match_quality,
    -- only a single distinct approved level is published, never a severity pick
    CASE
        WHEN classified.match_status = 'MATCHED' THEN classified.unique_flood_susceptibility_level
    END AS flood_susceptibility_level,
    CASE
        WHEN classified.match_status = 'MATCHED' THEN classified.unique_severity_rank
    END AS severity_rank,
    CASE
        WHEN classified.match_status = 'AMBIGUOUS' THEN 'MULTIPLE_APPROVED_FLOOD_LEVELS'
    END AS ambiguity_reason,
    CASE
        WHEN classified.match_status = 'MATCHED' THEN NULL
        WHEN classified.match_status = 'INVALID_SOURCE' THEN 'MISSING_STABLE_SOURCE_IDENTITY'
        WHEN classified.match_status = 'AMBIGUOUS' THEN 'MULTIPLE_APPROVED_FLOOD_LEVELS'
        WHEN classified.coordinate_status = 'MISSING_PAIR' THEN 'MISSING_COORDINATE_PAIR'
        WHEN classified.coordinate_status = 'PARTIAL_PAIR' THEN 'PARTIAL_COORDINATE_PAIR'
        WHEN classified.coordinate_status = 'OUT_OF_RANGE' THEN 'COORDINATE_OUT_OF_RANGE'
        ELSE 'NO_APPROVED_MGB_INTERSECTION'
    END AS exception_reason
FROM classified;

## 6. Build deterministic lineage and publish Table 15

**What:** publish exactly one row per canonical project with `CREATE OR REPLACE TABLE`. An exact rerun replaces the same logical rows and keys and never appends duplicates.

**Key:** `project_flood_map_key = SHA2(source_system | project_key | run_id, 256)`. This follows the Table 13 Silver SHA-256 convention. Gold owns its BIGINT surrogate keys later.

**Lineage kept on every row:**

- project snapshot, ingest run, optional publisher version, project run, project rule version, and project load time from `load_log`, as in Table 13
- MGB snapshot, ingest run, optional publisher version, and load time from Bronze `_ingested_at`, as in Table 14
- MGB mapping version and mapping source contract
- `classification_rule`, `classification_rule_version`, deterministic `run_id`, and `source_load_ts`

The optional publisher versions come only from `load_log`. They stay `NULL` when unavailable and are never fabricated.

**Expected validation:** the validator reproduces the key and run ID and proves mandatory lineage is present.

In [ ]:
%sql
CREATE OR REPLACE TABLE `02-silver`.silver_project_flood_map
USING DELTA
COMMENT 'One final flood-susceptibility classification per canonical project and deterministic project-flood mapping run'
AS
SELECT
    SHA2(CONCAT_WS(
        '|', decision.source_system, decision.project_key, selected.project_flood_run_id
    ), 256) AS project_flood_map_key,
    decision.source_system,
    decision.project_key,
    decision.contract_id,
    decision.latitude,
    decision.longitude,
    decision.coordinate_status,
    decision.flood_susceptibility_level,
    decision.severity_rank,
    decision.match_status,
    decision.match_method,
    decision.match_quality,
    CAST(decision.matched_polygon_count AS INT) AS matched_polygon_count,
    CAST(decision.matched_level_count AS INT) AS matched_level_count,
    decision.candidate_levels,
    decision.ambiguity_reason,
    decision.exception_reason,
    selected.classification_rule,
    selected.classification_rule_version,
    selected.project_source_snapshot_id,
    selected.project_source_ingest_run_id,
    selected.project_source_version,
    selected.project_run_id,
    selected.project_rule_version,
    selected.project_source_load_ts,
    selected.mgb_source_snapshot_id,
    selected.mgb_source_ingest_run_id,
    selected.mgb_source_version,
    selected.mgb_source_load_ts,
    selected.susceptibility_mapping_version,
    selected.mgb_mapping_source_version,
    selected.project_flood_run_id AS run_id,
    GREATEST(selected.project_source_load_ts, selected.mgb_source_load_ts) AS source_load_ts
FROM project_flood_decisions AS decision
CROSS JOIN selected_project_flood_versions AS selected;

## 7. Inspect compact runtime evidence

**What:** one grouped read of the published table by coordinate status, match outcome, reason, and level.

**Why:** reviewers see the coverage and the reasons for unresolved rows without rerunning the spatial join. The read uses the published Delta table, not the intermediate views, so no MGB parsing runs again.

**Expected validation:** these counts are evidence only. Nothing here is compared with a hardcoded expectation. Run `04_validation/08_validation_silver_project_flood_map.ipynb` next.

In [ ]:
%sql
SELECT
    coordinate_status,
    match_status,
    match_method,
    match_quality,
    exception_reason,
    flood_susceptibility_level,
    severity_rank,
    COUNT(*) AS project_rows,
    COUNT_IF(matched_polygon_count > 1) AS multiple_polygon_rows,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS project_percentage
FROM `02-silver`.silver_project_flood_map
GROUP BY ALL
ORDER BY project_rows DESC, coordinate_status, match_status, severity_rank;

# Summary

- **Table:** `02-silver.silver_project_flood_map`.
- **Grain:** one final project-to-flood classification per canonical `silver_project` row per deterministic project-flood mapping run. Every canonical project appears exactly once.
- **Primary key:** `project_flood_map_key`, a SHA-256 hash of source system, `project_key`, and `run_id`. Uniqueness also holds for `(source_system, contract_id, run_id)`.
- **Source dependencies:** `silver_project`, `01-bronze.flood_susceptibility`, `config_mgb_susceptibility_mapping`, and compact audit and validation evidence. No population, flood-control, Table 13, Table 14, or Gold data is read.
- **Coordinate contract:** the Table 13 screen and statuses. Only `VALID_PAIR` coordinates become points.
- **MGB contract:** only active approved mappings and Table 14's parser and geometry statuses. Only `MAPPED_APPROVED` and `USABLE_GEOMETRY` polygons take part.
- **Classification behavior:** a point inside one or more polygons of exactly one approved level is `MATCHED` with that level and rank. Same-level overlap stays `MATCHED` and keeps its polygon count.
- **Ambiguity behavior:** a point inside polygons of more than one approved level is `AMBIGUOUS` with no level or rank. No severity precedence is applied.
- **Unresolved behavior:** a usable point without an approved polygon, or a project without a usable point, is `UNMATCHED` with a controlled reason. It is never Low, zero, or `Unknown` risk.
- **Source accounting:** every selected MGB row lands in one mapping-by-geometry bucket, and the total equals the audited Bronze row count.
- **Idempotency:** `CREATE OR REPLACE TABLE` with a deterministic run ID and key. An exact rerun reproduces the same rows.
- **Validation handoff:** `04_validation/08_validation_silver_project_flood_map.ipynb` validates all seven data-quality attributes and persists evidence before its STOP gate.
- **Gold dependency:** future `fact_project_snapshot` resolves `MATCHED` levels to `dim_flood_susceptibility` and may route other rows to its governed unknown member. Gold must not repeat MGB mapping, parsing, point creation, spatial matching, or ambiguity resolution.
- **Known limitations:** projects without coordinates cannot be classified. Invalid MGB geometry is excluded, not repaired. The MGB publisher version may be unavailable.
- **Runtime:** requires Databricks execution on a runtime with native spatial SQL. The Esri path uses `ST_TRANSFORM`, which requires Databricks Runtime 17.1 or later. No runtime result is claimed by this notebook text.